In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import bbknn
import scrublet as scr
import gc

In [ ]:
pip show numpy

In [ ]:
# data dir
input_dir = " "
output_dir = " "


In [ ]:
os.listdir(input_dir)

In [ ]:
# pre work
file_list = os.listdir(input_dir)
# file_list = ["BRCA"]
doublet_all = pd.DataFrame()

# data run
for file in file_list:
    if file != "QC_data_stat.csv" and file != "QC_result.pdf" and file != "QC_result.png" and file != "QC_result.svg" and file != "QC_result.wmf":            
            print(f"################################## {file} process ##################################")

            ## data read
            scRNA_current = diopy.input.read_h5(file = f"{input_dir}/{file}/scRNA_merge.h5")

            ## dir create
            output_file = f"{output_dir}/{file}"
            os.makedirs(output_file, exist_ok=True)

            ## doublet detection
            sc.external.pp.scrublet(scRNA_current, batch_key="sample_ID",expected_doublet_rate=0.1)
            # if file == "MA":
            #     sc.external.pp.scrublet(scRNA_current, batch_key="sample_ID",expected_doublet_rate=0.1,n_prin_comps=10)
            # elif file == "liver_healthy":
            #     scRNA_current.obs["doublet_score"] = 0
            #     scRNA_current.obs["predicted_doublet"] = False
            # else:
            #     sc.external.pp.scrublet(scRNA_current, batch_key="sample_ID",expected_doublet_rate=0.1)
            doublet_current = scRNA_current.obs["predicted_doublet"].value_counts().reset_index()
            scRNA_current = scRNA_current[scRNA_current.obs.predicted_doublet==False, :]

            ## doublet stat
            doublet_current.columns = ["doublet_type", "count"]
            doublet_current["tumor_code"] = file
            doublet_all = pd.concat([doublet_all, doublet_current], ignore_index=True)

            ## data save
            diopy.output.write_h5(scRNA_current, file = f"{output_file}/scRNA_QC.h5",save_X=False)
            scRNA_current.write_h5ad(f"{output_file}/scRNA_QC.h5ad", compression="gzip")

# stat save
# doublet_all.to_csv(f"{output_dir}/scrublet_stat.csv", index=False)

In [ ]:
doublet_all